In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [ ]:
df = pd.read_csv('titanic_data_updated.csv')
df

In [ ]:
df.drop(['PassengerId','Name','Ticket'],axis=1,inplace=True)

In [ ]:
df

In [ ]:
df.isnull().sum()
# df.drop(['Cabin'],axis=1,inplace=True)

df

In [ ]:
X = df.drop(['Survived'],axis=1)

X

In [ ]:
y = df['Survived']

y

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
X_train
display(y_train)

In [ ]:
X_test
display(y_test)

In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

In [ ]:
#mean impute

age_mean = X_train['Age'].mean()

X_train['age_mean_imputor'] = X_train['Age'].fillna(age_mean)

X_test['age_mean_imputor'] = X_test['Age'].fillna(age_mean)


In [ ]:
#median impute

age_median = X_train['Age'].median()


X_train['age_median_imputor'] = X_train['Age'].fillna(age_median)

X_test['age_median_imputor'] = X_test['Age'].fillna(age_median)

X_train

In [ ]:
sns.kdeplot(data=X_train , x='age_mean_imputor')

In [ ]:
sns.kdeplot(data=X_train , x='age_median_imputor')

In [ ]:
from sklearn.impute import SimpleImputer

age_imputer = SimpleImputer(missing_values=np.nan , strategy='mean')

age_imputer.fit(X_train[['Age']])

X_train['Age'] = age_imputer.transform(X_train[['Age']]).ravel()




In [ ]:
X_train.isnull().sum()

X_train.drop(['age_mean_imputor','age_median_imputor'],axis=1,inplace=True)

X_train

In [ ]:
X_test['Age'] = age_imputer.transform(X_test[['Age']])

X_test.isnull().sum()

In [ ]:
X_test.drop(['age_mean_imputor','age_median_imputor'],axis=1,inplace=True)

X_test

In [ ]:
embarked_imputer = SimpleImputer(missing_values=np.nan , strategy='most_frequent')

embarked_imputer.fit(X_train[['Embarked']])

# transform both train and test data

X_train['Embarked']= embarked_imputer.transform(X_train[['Embarked']]).ravel()
X_test['Embarked'] = embarked_imputer.transform(X_test[['Embarked']]).ravel()




In [ ]:
X_train.isnull().sum()
X_test.isnull().sum()

In [ ]:
sns.countplot(data = X_train , x=X_train['Cabin'])

In [ ]:
cabin_imputer = SimpleImputer(missing_values=np.nan , strategy='constant',fill_value='Missing',add_indicator=True)

cabin_imputer.fit(X_train[['Cabin']])

# transform both train and test data

X_train[['Cabin','cabin_missing_indicator']]= cabin_imputer.transform(X_train[['Cabin']])
X_test[['Cabin','cabin_missing_indicator']] = cabin_imputer.transform(X_test[['Cabin']])

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_train.isnull().sum()

In [ ]:
X_test.isnull().sum()

# Outlier Detection

## Using Z-Score

In [ ]:
mean_of_age = X_train['Age'].mean()
std_of_age = X_train['Age'].std()

X_train['zscore_Age'] = (X_train['Age']-mean_of_age)/std_of_age


X_train


In [ ]:
outliers_age = X_train[abs(X_train['zscore_Age']) >3]

print(len(outliers_age))

display(outliers_age)

In [ ]:
mean_of_fare = X_train['Fare'].mean()
std_of_fare = X_train['Fare'].std()

X_train['zscore_Fare'] = (X_train['Fare']-mean_of_fare)/std_of_fare


X_train


In [ ]:
outliers_fare = X_train[abs(X_train['zscore_Fare']) >3]

print(len(outliers_fare))

display(outliers_fare)

## Outliers with IQR

In [ ]:
# age

age_Q1 = X_train['Age'].quantile(0.25)
age_Q3 = X_train['Age'].quantile(0.75)

age_IQR = age_Q3 - age_Q1

age_minimum = age_Q1 - 1.5 * age_IQR
age_maximum = age_Q3 + 1.5 * age_IQR

print(age_minimum , age_maximum)



In [ ]:
age_outlier_IQR = X_train[(X_train['Age']<age_minimum) | (X_train['Age']>age_maximum)]

print(len(age_outlier_IQR))
display(age_outlier_IQR)

In [ ]:
# fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)

fare_IQR = fare_Q3 - fare_Q1

fare_minimum = max(0,fare_Q1 - 1.5 * fare_IQR)
fare_maximum = fare_Q3 + 1.5 * fare_IQR

print(fare_minimum , fare_maximum) 
print(fare_Q1)
print(fare_Q3)



In [ ]:
fare_outlier_IQR = X_train[(X_train['Fare']<fare_minimum) | (X_train['Fare']>fare_maximum)]

print(len(fare_outlier_IQR))
display(fare_outlier_IQR)

In [ ]:
# z score outlier detection is better for age outliers

# zscore jader 3 er choto ba shoman shudhu matro tader kei amra train e rakhtesi
X_train = X_train[abs(X_train['zscore_Age']) <=3]

X_train

In [ ]:
minimum_zscore_fare = X_train[abs(X_train['zscore_Fare']) <=3]['Fare'].max()

print((minimum_zscore_fare))

In [ ]:
# iqr is better for fare

X_train['Fare']= X_train['Fare'].clip(fare_minimum , fare_maximum)

X_train['Fare'].max()

In [ ]:
X_train

In [ ]:
X_train.drop(['zscore_Age','zscore_Fare'],axis=1 , inplace=True)

X_train